# Microglia motility — analysis tutorial (Step 8–9)

**What this notebook is.** A walk-through of the Python half of the pipeline: it takes the two aggregated TrackMate CSVs (`all_spots.csv`, `all_tracks.csv`) and turns them into the count and motility figures, then adds two new analyses — **mean squared displacement (MSD)** and **per-image trajectory maps**.

**How to read it.** Every code cell has a markdown cell above it explaining *what* it does, *why* it's done that way, and the specific pandas / matplotlib moves used — so you can rebuild it yourself. Comments inside the code explain the lines; the markdown explains the ideas.

---

### The data model (read this once)

Each **spot** = one microglia detection in one frame. Each **track** = one cell followed across frames (a chain of spots sharing a `TRACK_ID`). Two files:

| File | One row = | Key columns we use |
|------|-----------|--------------------|
| `all_spots.csv` | one detection in one frame | `POSITION_X/Y`, `FRAME`, `POSITION_T`, `MEDIAN_INTENSITY_CH5` (region), `CIRCULARITY`, `TRACK_ID` |
| `all_tracks.csv` | one whole track | `NUMBER_SPOTS`, `NUMBER_GAPS`, `CONFINEMENT_RATIO`, `MEAN_STRAIGHT_LINE_SPEED`, `TRACK_MEDIAN_SPEED` |

The region label is carried **per spot** in `MEDIAN_INTENSITY_CH5` — the value of channel 5 (the region map your Step 04 macro paints) sampled under each cell. Verified against your current macro and sample data, the values are:

| CH5 value | Region | Notes |
|-----------|--------|-------|
| `0` | outside annotated tissue | non–spinal-cord cells; usually excluded |
| `1` | uninjured side | |
| `2` | injured side (penumbra) | |
| `3` | injury core | the tight ROI inside the injured side |

> **Two granularities, used deliberately.** For *counts* we collapse to **injured vs uninjured** (core folds into injured, because it *is* the injured side and splitting it would undercount). For *motility* we keep the full **core / penumbra / uninjured** gradient, because the whole question is whether cells behave differently as you move toward the core.

---

### Before you trust the numbers — three judgement calls baked in

1. **Why median, not mean, for the region label.** A cell's footprint can straddle a region boundary. `MEAN_INTENSITY_CH5` then comes back fractional (e.g. 1.7) and is meaningless. `MEDIAN_INTENSITY_CH5` returns the value covering *most* of the cell — a clean integer. Always use the median for label-map channels. (Your sample data confirms it: the mean column is full of fractions like 0.28; the median column is exactly 0/1/2/3.)
2. **Unlinked detections.** ~11% of spots in the sample have no `TRACK_ID` (TrackMate detected them but couldn't link them into a track). They are real cells for *counting* but cannot contribute to *motility*. Cell 3 reports how many; Cell 4 lets you choose whether counts include them.
3. **Cadence differs between images** (30 vs 40 min/frame here). So "frame 5" is a different real time in different fish. Every time axis below is converted to **minutes**, never left in frames.


## Cell 1 — Load the two CSVs into the session

In Colab, `files.upload()` opens a browser dialog and writes whatever you pick into `/content/`. You can select **both** CSVs at once. The returned `uploaded` dict maps filename → bytes; we only need the keys (the filenames), which we use in the next cell.

If you're running locally instead of Colab, skip this cell and just set the two paths by hand in Cell 2.

In [ ]:
# Run this, then pick BOTH all_spots.csv AND all_tracks.csv in the dialog.
from google.colab import files
uploaded = files.upload()                 # browser dialog; multi-select is allowed
print("\nUploaded:", list(uploaded.keys()))

## Cell 2 — Read them into DataFrames (the 4-row header trick)

TrackMate writes **four header rows**, not one:

```
row 0:  ALL_CAPS machine keys    <- the real column names we want
row 1:  Human readable names     \
row 2:  Short abbreviations       }  decoration we must throw away
row 3:  Units                    /
```

`pd.read_csv` applies `skiprows` **first**, then reads `header` from what's left. So:
- `header=0` → use row 0 (the keys) as column names
- `skiprows=[1,2,3]` → drop the three decoration rows so they aren't read as data

We find which file is which **by name** (`"spots"`/`"tracks"` in the filename) so upload order doesn't matter. Note: these files were concatenated from 9 source images by your Step 07 export, which prepends a `SOURCE_IMAGE` column — that's the one non-stock column.

In [ ]:
import pandas as pd

# If you're NOT in Colab, replace the next two lines with literal paths, e.g.:
#   spots_file, tracks_file = "all_spots.csv", "all_tracks.csv"
spots_file  = next(f for f in uploaded if "spots"  in f.lower())
tracks_file = next(f for f in uploaded if "tracks" in f.lower())

# The 4-row-header read pattern — memorise this; you'll use it for every TrackMate CSV.
spots  = pd.read_csv(spots_file,  header=0, skiprows=[1, 2, 3])
tracks = pd.read_csv(tracks_file, header=0, skiprows=[1, 2, 3])

print(f"spots  : {spots.shape[0]:>5} rows x {spots.shape[1]} cols")
print(f"tracks : {tracks.shape[0]:>5} rows x {tracks.shape[1]} cols")
print(f"source images: {spots['SOURCE_IMAGE'].nunique()}")

## Cell 3 — Integrity checks (look before you leap)

No analysis here — this cell only *verifies the assumptions the rest of the notebook depends on* and adds one derived column (`is_manual`). Read all five printouts before trusting anything downstream. New techniques introduced:

- **`value_counts(dropna=False)`** — frequency table that also counts missing values, so nothing hides.
- **`.groupby([...]).ngroups`** — counts distinct *combinations* of keys. We use it to show why `TRACK_ID` alone is not a cell identity.
- **A boolean column from a comparison** (`spots["CIRCULARITY"] == 1.0`) — vectorised, no loop.

**Why `(SOURCE_IMAGE, TRACK_ID)` is the true cell key.** Each image restarts `TRACK_ID` from 0, so track 4 in fish A and track 4 in fish B are different cells that would collide if you grouped on `TRACK_ID` alone. We group on **both** columns everywhere.

**The `is_manual` flag.** Spots you placed by hand during curation are perfect circles (`CIRCULARITY == 1.0` exactly, radius 5). Automated MaskDetector spots inherit the real cell outline, so circularity < 1. We *flag* rather than drop them: a hand-placed spot is a real cell (fine for counting and for position-based motility) but its *shape* is a placeholder (exclude it from any morphology metric).

**The unlinked-detection report (item 5)** is the new one — it surfaces the spots with no `TRACK_ID` so you can see how many there are before deciding what counts should include.

In [ ]:
# --- 1. Region label lives PER-SPOT in MEDIAN_INTENSITY_CH5 ----------------
#   0 = outside tissue, 1 = uninjured, 2 = injured (penumbra), 3 = injury core.
print("1) Region label counts (MEDIAN_INTENSITY_CH5):")
print(spots["MEDIAN_INTENSITY_CH5"].value_counts(dropna=False).sort_index(), "\n")

# --- 2. Manual-curation flag, read off the morphology ---------------------
spots["is_manual"] = spots["CIRCULARITY"] == 1.0
n_manual = int(spots["is_manual"].sum())
print(f"2) Manual spots: {n_manual} / {len(spots)} ({100*n_manual/len(spots):.1f}%)\n")

# --- 3. Track identity = (SOURCE_IMAGE, TRACK_ID), NOT TRACK_ID alone ------
print(f"3) distinct TRACK_ID alone        : {spots['TRACK_ID'].nunique()}")
print(f"   distinct (SOURCE_IMAGE,TRACK_ID): "
      f"{spots.groupby(['SOURCE_IMAGE','TRACK_ID']).ngroups}  <- true track count\n")

# --- 4. Sampling interval differs between images --------------------------
#   POSITION_T = absolute time in seconds; FRAME = integer index.
#   interval (min) = last_time_seconds / (n_frames - 1) / 60.
per_image = (spots.groupby("SOURCE_IMAGE")
                  .agg(n_frames=("FRAME", lambda s: int(s.max()) + 1),
                       max_T_s =("POSITION_T", "max")))
per_image["interval_min"] = (per_image["max_T_s"]
                             / (per_image["n_frames"] - 1) / 60).round(1)
print("4) Per-image sampling:")
print(per_image, "\n")

# --- 5. Unlinked detections (NEW) -----------------------------------------
#   Spots with a missing TRACK_ID were detected but never linked into a track.
#   They are real cells for COUNTING but cannot contribute to MOTILITY
#   (a single isolated point has no step, speed, or displacement).
#   groupby() silently DROPS rows whose key is NaN, so these never reach the
#   track-level analysis anyway -- but they DO reach the per-frame counts,
#   so it's worth seeing the size of the group before Cell 4.
n_unlinked = int(spots["TRACK_ID"].isna().sum())
print(f"5) Unlinked detections (no TRACK_ID): {n_unlinked} / {len(spots)} "
      f"({100*n_unlinked/len(spots):.1f}%)")
print("   per image:")
print(spots[spots['TRACK_ID'].isna()].groupby('SOURCE_IMAGE').size()
      .rename('n_unlinked').to_string())

## Cell 4 — Microglia counts over real time, one panel per image

**The biological question (Q1):** are there more microglia on the injured side, and how does that change over time?

**Counting recipe.** One spot = one cell in one frame, so the number of cells in a (image, frame, region) bucket is just the number of rows in it: `groupby([...]).size()`. To draw a line we need a value at *every* frame including empty ones, so we `pivot` region into columns and `reindex` to the full frame range, filling gaps with 0 (a 0 reads as a point on the line, not a break).

**Two region mappings** are built here from the same per-spot label:
- `region` (simple, for counts): core folds **into** injured → `{0:outside, 1:uninjured, 2:injured, 3:injured}`
- `region_fine` (gradient, for motility later): `{0:outside, 1:uninjured, 2:penumbra, 3:core}`

**Plotting choices**, all deliberate: x = real minutes (frame × that image's interval); identical y-range on every panel so panels are comparable at a glance; a light tint on any image whose cadence differs from the majority, so the odd one out is obvious.

**The new knob — `COUNT_ONLY_TRACKED`.** Set it `True` to count only spots that made it into a track (drops the unlinked detections from Cell 3, item 5); `False` keeps every detection. Default `False` preserves prior behaviour; flip it to see how sensitive the counts are to unlinked spots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ---- knobs ---------------------------------------------------------------
TICK_MIN          = 60        # x-tick spacing (min)
NCOLS             = 2         # subplot columns
COUNT_ONLY_TRACKED = False    # True -> exclude unlinked detections from counts
COLORS = {"injured": "red", "uninjured": "blue", "outside": "lightgrey"}
TINT   = "#fff5e6"            # background tint for odd-cadence images
ORDER  = ["injured", "uninjured", "outside"]

# ---- two region granularities from one per-spot label --------------------
REGION      = {0.0: "outside", 1.0: "uninjured", 2.0: "injured",   3.0: "injured"}
REGION_FINE = {0.0: "outside", 1.0: "uninjured", 2.0: "penumbra",  3.0: "core"}
spots["region"]      = spots["MEDIAN_INTENSITY_CH5"].map(REGION)
spots["region_fine"] = spots["MEDIAN_INTENSITY_CH5"].map(REGION_FINE)

# ---- choose which spots feed the counts ----------------------------------
count_src = spots[spots["TRACK_ID"].notna()] if COUNT_ONLY_TRACKED else spots

counts = count_src.groupby(["SOURCE_IMAGE", "FRAME", "region"]).size()
YMAX   = int(counts.max()) + 1                      # shared y-axis

intervals     = per_image["interval_min"]
mode_interval = intervals.mode().iloc[0]            # the majority cadence
XMAX = float((per_image["n_frames"] - 1).mul(intervals).max())  # longest image, min

images = sorted(spots["SOURCE_IMAGE"].unique())
nrows  = int(np.ceil(len(images) / NCOLS))
fig, axes = plt.subplots(nrows, NCOLS, figsize=(7 * NCOLS, 2.6 * nrows))
axes = axes.ravel()

for ax, img in zip(axes, images):
    interval = intervals[img]
    sub      = count_src[count_src["SOURCE_IMAGE"] == img]
    frames   = np.arange(int(sub["FRAME"].max()) + 1)
    t_min    = frames * interval                    # real-time x for THIS image

    # counts per region, reindexed so empty frames read 0 (a line, not a gap)
    piv = (sub.groupby(["FRAME", "region"]).size()
              .unstack("region")
              .reindex(index=frames, columns=ORDER)
              .fillna(0))
    for grp in ORDER:
        ax.plot(t_min, piv[grp], marker="o", ms=3, lw=1.5,
                color=COLORS[grp], label=grp)

    if interval != mode_interval:
        ax.set_facecolor(TINT)
    ax.set_title(f"{img.split('_')[2]}  ({interval:.0f} min)", fontsize=9)
    ax.set_xlabel("Time (min)"); ax.set_ylabel("microglia count")
    ax.set_ylim(0, YMAX); ax.set_xlim(0, XMAX)
    ax.set_xticks(np.arange(0, XMAX + 1, TICK_MIN))
    ax.tick_params(labelsize=8)

for ax in axes[len(images):]:
    ax.axis("off")

handles = [plt.Line2D([], [], color=COLORS[g], marker="o", ms=4, label=g) for g in ORDER]
handles.append(Patch(facecolor=TINT, edgecolor="grey",
                     label=f"odd sampling (\u2260 {mode_interval:.0f} min)"))
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False)
fig.suptitle("Microglia count per region over time, by image", fontsize=12)
fig.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()

## Cell 5 — Pool the images: mean ± SD count per region over time

Cell 4 shows every fish; this distils them into one curve per region with a spread band. The only hard part is that the fish have different cadences and lengths, so we can't just average "frame 5" across fish. We fix it by **binning onto a common minute grid**:

1. For each fish, assign every frame to a time bin (`(minutes // BIN) * BIN`) and take the **mean count per bin** (so a 30-min and a 40-min fish both contribute one number per bin).
2. *Then* across fish, take **mean, SD and n** per bin.
3. Drop bins where fewer than `MIN_IMAGE_FRAC` of fish contribute — the tail of a timelapse is averaged over very few animals and gets noisy.

New techniques: a two-stage groupby (within-fish, then across-fish), and a `MultiIndex` column frame from `.agg(["mean","std","count"])` — you index it as `summary[(region, "mean")]`.

> **Caveat to keep in mind:** these are **raw counts**, not normalised by region area. If the injured side covers more (or less) tissue than the uninjured side, some of the count difference is just area. Cell 6 repeats the warning; a proper fix is to divide by region area in µm² once you export it.

In [ ]:
BIN_MIN        = 60      # common time-bin width (min)
SHOW_OUTSIDE   = False   # include the 'outside tissue' region?
MIN_IMAGE_FRAC = 0.75    # drop bins where < this fraction of images contribute

# --- 1) per-image, per-bin MEAN count per region --------------------------
per_fish_bin = []
for img in images:
    interval = intervals[img]
    sub      = count_src[count_src["SOURCE_IMAGE"] == img]
    frames   = np.arange(int(sub["FRAME"].max()) + 1)
    piv = (sub.groupby(["FRAME", "region"]).size()
              .unstack("region").reindex(index=frames, columns=ORDER).fillna(0))
    piv["bin"] = ((frames * interval) // BIN_MIN * BIN_MIN).astype(int)
    g = piv.groupby("bin")[ORDER].mean()
    g["SOURCE_IMAGE"] = img
    per_fish_bin.append(g.reset_index())
per_fish_bin = pd.concat(per_fish_bin, ignore_index=True)

# --- 2) across images: mean, SD, n per bin --------------------------------
summary  = per_fish_bin.groupby("bin")[ORDER].agg(["mean", "std", "count"])
n_perbin = summary[(ORDER[0], "count")]
print("images contributing per time bin:")
print(n_perbin.rename("n_images"))

# --- 3) apply the contribution cutoff -------------------------------------
keep    = n_perbin >= MIN_IMAGE_FRAC * len(images)
summary = summary[keep]
print(f"\nkept {keep.sum()}/{len(keep)} bins "
      f"(>= {MIN_IMAGE_FRAC:.0%} of {len(images)} images)")

# --- 4) plot --------------------------------------------------------------
plot_regions = ORDER if SHOW_OUTSIDE else ["injured", "uninjured"]
fig, ax = plt.subplots(figsize=(10, 5))
x = summary.index
for grp in plot_regions:
    m = summary[(grp, "mean")]
    s = summary[(grp, "std")].fillna(0)
    ax.plot(x, m, marker="o", color=COLORS[grp], label=grp)
    ax.fill_between(x, (m - s).clip(lower=0), m + s, color=COLORS[grp], alpha=0.2)
ax.set_xlabel("Time (min)")
ax.set_ylabel(f"mean microglia count  (\u00b1 SD, {BIN_MIN}-min bins)")
ax.set_xticks(np.arange(0, x.max() + 1, TICK_MIN))
ax.set_ylim(bottom=0)
ax.legend(title=f"region (n={len(images)} images)")
ax.set_title("Cross-image mean microglia count per region over time")
fig.tight_layout(); plt.show()

## Cell 6 — Paired per-fish comparison: uninjured vs injured

One summary number per fish per region, with the two regions of the same fish joined by a line so you can see the *within-animal* difference (a paired design is far more sensitive than comparing two clouds of points).

**Metric:** mean microglia present per frame over the whole timelapse = (total spots in region) / (number of frames). Dividing by frame count makes it independent of how long or how fast each fish was imaged.

The black marker is the group mean with a **95% confidence interval** (mean ± t* × SEM). This is *exploratory* — the CI describes the mean's precision, it is **not** a hypothesis test. We deliberately don't run a p-value here; that comes later with a model that respects the fish-level grouping.

New techniques: `.div(..., axis=0)` to divide each row by a per-row value, and `scipy.stats.t.ppf` for the t multiplier.

In [ ]:
from scipy import stats   # only for the t-multiplier of the CI -- not a test

totals = (count_src[count_src["region"].isin(["uninjured", "injured"])]
          .groupby(["SOURCE_IMAGE", "region"]).size()
          .unstack("region").reindex(columns=["uninjured", "injured"]).fillna(0))
per_fish = totals.div(per_image["n_frames"], axis=0)     # mean cells/frame per region

n     = len(per_fish)
means = per_fish[["uninjured", "injured"]].mean()
sems  = per_fish[["uninjured", "injured"]].sem()         # std / sqrt(n), ddof=1
ci95  = stats.t.ppf(0.975, df=n - 1) * sems

fig, ax = plt.subplots(figsize=(5, 6))
for _, row in per_fish.iterrows():                       # one connector per fish
    ax.plot([0, 1], [row["uninjured"], row["injured"]], "-", color="grey", alpha=0.5, zorder=1)
ax.scatter([0]*n, per_fish["uninjured"], color=COLORS["uninjured"], zorder=2)
ax.scatter([1]*n, per_fish["injured"],   color=COLORS["injured"],   zorder=2)
ax.errorbar([0, 1], means.values, yerr=ci95.values,
            fmt="-o", color="black", lw=2.5, capsize=6, zorder=3, label="mean \u00b1 95% CI")
ax.set_title(f"Mean microglia per frame, full timelapse  (n = {n} fish)")
ax.set_xticks([0, 1]); ax.set_xticklabels(["uninjured", "injured"])
ax.set_ylabel("mean microglia per frame")
ax.set_xlim(-0.4, 1.4); ax.set_ylim(bottom=0); ax.legend()
fig.tight_layout(); plt.show()

## Cell 7 — Build the per-track motility table

This is the workhorse table for the motility half. TrackMate's `tracks` file already has kinematics (speed, confinement, displacement); we **add** the things it doesn't know about and join them on, one row per `(SOURCE_IMAGE, TRACK_ID)`:

1. **Region composition** — what fraction of each track's spots fell in core / penumbra / uninjured / outside (`value_counts().unstack()` → divide by row totals).
2. **A single region label per track**, via a rule you choose: `majority` (where it spent most frames), `end` (where it finished), or `deepest` (the most-injured zone it ever touched). Different biological questions want different rules — that's why it's a knob.
3. **Directionality toward the injury.** TrackMate only gives displacement *magnitude*. But because Step 04 standardises every image to **injury-at-top**, "toward injury" = "moving to smaller Y". So we compute net Y travel from the first and last spot ourselves. `directedness` = (toward-injury distance) / (net displacement) lives in [−1, 1]: +1 = straight at the injury, −1 = straight away, ≈0 = sideways.

> **Why the orientation assumption is safe:** your `04_draw_injury.ijm` flips each stack so injury ends up at the top *before* tracking, and TrackMate runs on the flipped pixels — so the `POSITION_Y` in the CSV is already in the standardised frame. If you ever change that convention, flip `INJURY_AT_TOP`.

New techniques: `.value_counts().unstack(fill_value=0)`, chained `.merge(...)` to assemble a wide table, `np.hypot` for Euclidean distance, and boolean feature columns (`reaches_core`, `is_migrator`).

Quality gate: drop tracks shorter than `MIN_SPOTS` (kinematics from 2–3 points are noise). `MAX_GAPS_ALLOWED` is there if you want gap-free tracks only.

In [ ]:
# ---- knobs ---------------------------------------------------------------
MIN_SPOTS        = 5          # drop tracks shorter than this
MAX_GAPS_ALLOWED = None       # None = keep all; 0 = gap-free only
REGION_RULE      = "majority" # "majority" | "end" | "deepest"
INJURY_AT_TOP    = True       # Step 04 puts injury at top -> toward-injury = decreasing Y

KEYS       = ["SOURCE_IMAGE", "TRACK_ID"]
FINE_ORDER = ["core", "penumbra", "uninjured", "outside"]

# ---- 1) per-track region composition (FINE granularity) ------------------
reg_counts = (spots.groupby(KEYS)["region_fine"]
                   .value_counts().unstack(fill_value=0)
                   .reindex(columns=FINE_ORDER, fill_value=0))
reg_frac = reg_counts.div(reg_counts.sum(axis=1), axis=0)
reg_frac.columns = [f"frac_{c}" for c in reg_frac.columns]

# ---- 1b) one region label per track, per the chosen rule -----------------
if REGION_RULE == "majority":
    region_track = reg_counts.idxmax(axis=1).rename("region_track")
elif REGION_RULE == "end":
    region_track = (spots.sort_values("FRAME").groupby(KEYS).tail(1)
                         .set_index(KEYS)["region_fine"].rename("region_track"))
elif REGION_RULE == "deepest":
    region_track = (spots.groupby(KEYS)["MEDIAN_INTENSITY_CH5"].max()
                         .map(REGION_FINE).rename("region_track"))
else:
    raise ValueError("REGION_RULE must be 'majority', 'end' or 'deepest'")

# ---- 2) fraction of each track's spots that were MANUAL -------------------
frac_manual = spots.groupby(KEYS)["is_manual"].mean().rename("fraction_manual")

# ---- 3) DIRECTIONALITY toward the injury (computed from spots) -----------
ordered = spots.sort_values("FRAME")
first   = ordered.groupby(KEYS)[["POSITION_X", "POSITION_Y"]].first()
last    = ordered.groupby(KEYS)[["POSITION_X", "POSITION_Y"]].last()
dxy = last - first
net_disp = np.hypot(dxy["POSITION_X"], dxy["POSITION_Y"]).rename("net_disp_um")
toward   = (-dxy["POSITION_Y"] if INJURY_AT_TOP else dxy["POSITION_Y"]).rename("toward_injury_um")
directedness = (toward / net_disp).replace([np.inf, -np.inf], np.nan).rename("directedness")
direction = pd.concat([net_disp, toward, directedness], axis=1)

# ---- 4) merge everything onto TrackMate's track features -----------------
per_track = (tracks
             .merge(region_track.reset_index(), on=KEYS)
             .merge(reg_frac.reset_index(),     on=KEYS)
             .merge(frac_manual.reset_index(),  on=KEYS)
             .merge(direction.reset_index(),    on=KEYS))

per_track["frac_injured_side"] = per_track["frac_core"] + per_track["frac_penumbra"]
per_track["reaches_core"]      = per_track["frac_core"] > 0
per_track["is_migrator"]       = ((per_track["frac_uninjured"] > 0) &
                                  (per_track["frac_injured_side"] > 0))

# ---- 5) quality gate -----------------------------------------------------
n_before = len(per_track)
keep = per_track["NUMBER_SPOTS"] >= MIN_SPOTS
if MAX_GAPS_ALLOWED is not None:
    keep &= per_track["NUMBER_GAPS"] <= MAX_GAPS_ALLOWED
per_track = per_track[keep].copy()

# ---- 6) report -----------------------------------------------------------
print(f"tracks: {n_before} -> {len(per_track)} kept "
      f"(MIN_SPOTS={MIN_SPOTS}, MAX_GAPS_ALLOWED={MAX_GAPS_ALLOWED}, REGION_RULE='{REGION_RULE}')")
print("\nregion_track counts (kept):")
print(per_track["region_track"].value_counts())
print(f"\nmigrators (uninjured <-> injured side): {int(per_track['is_migrator'].sum())}")
print(f"tracks reaching the core: {int(per_track['reaches_core'].sum())}")

per_track[KEYS + ["region_track", "frac_uninjured", "frac_penumbra", "frac_core",
                  "is_migrator", "reaches_core", "fraction_manual", "NUMBER_SPOTS",
                  "toward_injury_um", "directedness", "CONFINEMENT_RATIO",
                  "MEAN_STRAIGHT_LINE_SPEED"]].head(12)

## Cell 8 — Speed vs confinement, per image and pooled

A two-variable portrait of each track:
- **x = `MEAN_STRAIGHT_LINE_SPEED`** (net displacement ÷ time) — robust to cadence because it's a straight-line rate, not a sum of jittery steps. Converted to µm/hour for readability.
- **y = `CONFINEMENT_RATIO`** (displacement ÷ path length): 0 = went nowhere (confined), 1 = straight line (directed).
- **colour = the track's fine region**; **marker size ∝ track length** so short, less-trustworthy tracks are visibly small.

Your hypothesis ("cells park in the core") predicts **dark-red dots in the bottom-left** — low speed *and* low confinement. Part A shows each image; Part B pools them so you can see the overall cloud.

In [ ]:
SPEED_COL   = "MEAN_STRAIGHT_LINE_SPEED"
TO_PER_HOUR = 3600.0     # TrackMate speed is um/s -> x3600 = um/hour
NCOLS       = 2
MOT_ORDER  = ["core", "penumbra", "uninjured"]
MOT_COLORS = {"core": "darkred", "penumbra": "red", "uninjured": "blue"}

plot_df    = per_track[per_track["region_track"].isin(MOT_ORDER)]
mot_images = sorted(plot_df["SOURCE_IMAGE"].unique())
xmax       = plot_df[SPEED_COL].max() * TO_PER_HOUR * 1.05

# --- Part A: one panel per image ---
nrows = int(np.ceil(len(mot_images) / NCOLS))
fig, axes = plt.subplots(nrows, NCOLS, figsize=(7 * NCOLS, 3 * nrows))
axes = axes.ravel()
for ax, img in zip(axes, mot_images):
    sub = plot_df[plot_df["SOURCE_IMAGE"] == img]
    for grp in MOT_ORDER:
        g = sub[sub["region_track"] == grp]
        ax.scatter(g[SPEED_COL] * TO_PER_HOUR, g["CONFINEMENT_RATIO"],
                   s=10 + 4 * g["NUMBER_SPOTS"], color=MOT_COLORS[grp],
                   alpha=0.7, edgecolor="k", linewidth=0.3, label=grp)
    ax.set_title(img.split("_")[2], fontsize=9)
    ax.set_xlabel("straight-line speed (\u00b5m/hour)"); ax.set_ylabel("confinement ratio")
    ax.set_xlim(0, xmax); ax.set_ylim(0, 1.02)
for ax in axes[len(mot_images):]:
    ax.axis("off")
handles = [plt.Line2D([], [], marker="o", ls="", color=MOT_COLORS[g], label=g) for g in MOT_ORDER]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False)
fig.suptitle("Per-image microglia motility: speed vs confinement", fontsize=14)
fig.tight_layout(rect=[0, 0.04, 1, 0.97]); plt.show()

# --- Part B: pooled ---
fig_sum, ax_sum = plt.subplots(figsize=(8, 6))
for grp in MOT_ORDER:
    g = plot_df[plot_df["region_track"] == grp]
    ax_sum.scatter(g[SPEED_COL] * TO_PER_HOUR, g["CONFINEMENT_RATIO"],
                   s=15 + 5 * g["NUMBER_SPOTS"], color=MOT_COLORS[grp],
                   alpha=0.6, edgecolor="k", linewidth=0.4, label=grp)
ax_sum.set_title(f"Summary: pooled motility across all {len(mot_images)} images", fontsize=12)
ax_sum.set_xlabel("straight-line speed (\u00b5m/hour)"); ax_sum.set_ylabel("confinement ratio")
ax_sum.set_xlim(0, xmax); ax_sum.set_ylim(0, 1.02); ax_sum.legend(title="region")
fig_sum.tight_layout(); plt.show()

## Cell 9 — Pooled per-region motility (one metric, one dot per track)

The simplest aggregate: pool every track and compare a single metric across the injury gradient (uninjured → penumbra → core). One jittered dot per track; the black bar is the **median** with the **inter-quartile range** (median + IQR, not mean + SD, because track metrics are skewed and have outliers).

Swap `METRIC` to ask different questions — it's just a column name (`CONFINEMENT_RATIO`, `directedness`, `toward_injury_um`, …). Set `SCALE = 3600` for speed columns (µm/s → µm/hour) and `1.0` otherwise.

> **Honest caveat:** pooling treats every track as independent and ignores that tracks are nested within fish. Fine for exploration; for a final statistic you'd use a mixed-effects model with fish as a random effect.

In [ ]:
METRIC     = "TRACK_MEDIAN_SPEED"   # try: CONFINEMENT_RATIO, directedness, toward_injury_um
SCALE      = 3600.0                 # um/s -> um/hour (use 1.0 for non-speed metrics)
YLABEL     = "median step speed (\u00b5m/hour)"
REGIONS    = ["uninjured", "penumbra", "core"]
MOT_COLORS = {"core": "darkred", "penumbra": "red", "uninjured": "blue"}

pdf = per_track[per_track["region_track"].isin(REGIONS)].copy()
pdf["_y"] = pdf[METRIC] * SCALE

rng = np.random.default_rng(0)       # reproducible jitter
fig, ax = plt.subplots(figsize=(6, 5))
for i, grp in enumerate(REGIONS):
    g = pdf.loc[pdf["region_track"] == grp, "_y"].dropna()
    ax.scatter(i + rng.uniform(-0.12, 0.12, len(g)), g, color=MOT_COLORS[grp],
               alpha=0.7, edgecolor="k", linewidth=0.3, zorder=2)
    if len(g):
        med = g.median(); q1, q3 = g.quantile([0.25, 0.75])
        ax.plot([i-0.22, i+0.22], [med, med], color="k", lw=2.5, zorder=3)
        ax.plot([i, i], [q1, q3], color="k", lw=1.2, zorder=3)
        ax.text(i+0.28, med, f"n={len(g)}", va="center", fontsize=8)
ax.axhline(0, color="grey", lw=0.5)
ax.set_xticks(range(len(REGIONS))); ax.set_xticklabels(REGIONS)
ax.set_ylabel(YLABEL); ax.set_xlim(-0.5, len(REGIONS)-0.5)
ax.set_title(f"Pooled per-region motility: {METRIC}  (dot = track, n={len(pdf)})")
fig.tight_layout(); plt.show()

## Cell 10 — Step-speed distributions per region (ridge plot)

Cell 9 reduced each track to one number. This goes finer: it looks at **every individual step** (frame-to-frame jump) as a speed, and compares the *shape* of the step-speed distribution across regions. If cells park in the core, the core ridge should sit shifted toward low speed.

**Computing a step.** For each track, sorted by frame, a step is the move between consecutive detections: distance `np.hypot(dx, dy)` over elapsed time `dt`. `.diff()` within each group gives those deltas; the first step of each track is `NaN` (nothing before it) and gets dropped. Each step is attributed to the region at its **start** (`region_fine.shift()`).

**A real decision — gap steps.** When a track has a gap, a "step" may span 2+ frames. Its straight-line distance misses the wiggle in between, so it **underestimates** speed. `EXCLUDE_GAP_STEPS` controls this:
- `True` (recommended for the motility distribution) → keep only true 1-frame steps (`dframe == 1`). This matches the principle that gapped steps are kept for QC but excluded from motility.
- `False` → keep them (speed is still distance/real-time, just biased low across gaps).

I've defaulted it to `True` here so the distribution is clean; flip it to see the difference.

**The drawing.** A KDE (smoothed histogram) per region, stacked as overlapping "ridges". `SMOOTHING` is the KDE bandwidth — smaller = bumpier/more detail, larger = smoother. Dashed tick = each region's median.

In [ ]:
from scipy.stats import gaussian_kde

REGIONS    = ["uninjured", "penumbra", "core"]   # top -> bottom of the ridge
MOT_COLORS = {"core": "darkred", "penumbra": "red", "uninjured": "blue"}
EXCLUDE_GAP_STEPS = True   # True = 1-frame steps only (recommended for motility)
OVERLAP    = 1.4           # ridge height vs row spacing
SMOOTHING  = 0.3           # KDE bandwidth: smaller = more detail, larger = smoother

# ---- recompute per-step speeds (self-contained) --------------------------
s   = spots.merge(per_track[KEYS].drop_duplicates(), on=KEYS, how="inner")
s   = s.sort_values(KEYS + ["FRAME"])
grp = s.groupby(KEYS, sort=False)
s["dist_um"]      = np.hypot(grp["POSITION_X"].diff(), grp["POSITION_Y"].diff())
s["dt_s"]         = grp["POSITION_T"].diff()
s["dframe"]       = grp["FRAME"].diff()
s["region_start"] = grp["region_fine"].shift()
steps = s.dropna(subset=["dist_um", "dt_s", "region_start"])
steps = steps[steps["dt_s"] > 0]
if EXCLUDE_GAP_STEPS:
    steps = steps[steps["dframe"] == 1]
steps = steps.assign(speed_um_hour=steps["dist_um"] / steps["dt_s"] * 3600.0)

# ---- build the ridges ----------------------------------------------------
vmax = np.nanpercentile(steps["speed_um_hour"], 99)
xs   = np.linspace(0, vmax, 300)
n    = len(REGIONS)
dens, counts_per_ridge = {}, {}
for r in REGIONS:
    v = steps.loc[steps["region_start"] == r, "speed_um_hour"].dropna().values
    counts_per_ridge[r] = len(v)
    dens[r] = gaussian_kde(v, bw_method=SMOOTHING)(xs) if (len(v) >= 2 and np.std(v) > 0) else np.zeros_like(xs)
gmax = max((d.max() for d in dens.values()), default=1.0) or 1.0

fig, ax = plt.subplots(figsize=(8, 5))
for idx, r in enumerate(REGIONS):
    base = n - 1 - idx
    d    = dens[r] / gmax * OVERLAP
    z    = idx + 2
    ax.fill_between(xs, base, base + d, color=MOT_COLORS[r], alpha=0.8, zorder=z)
    ax.plot(xs, base + d, color="k", lw=0.8, zorder=z)
    v = steps.loc[steps["region_start"] == r, "speed_um_hour"].dropna()
    if len(v):
        ax.plot([v.median()] * 2, [base, base + 0.18 * OVERLAP], color="k", lw=1.4, ls="--", zorder=z + 0.1)
ax.set_yticks([n - 1 - i for i in range(n)])
ax.set_yticklabels([f"{r}\n(steps={counts_per_ridge[r]})" for r in REGIONS])
for tick, r in zip(ax.get_yticklabels(), REGIONS):
    tick.set_color(MOT_COLORS[r]); tick.set_fontsize(9)
ax.set_ylim(-0.2, n - 1 + OVERLAP + 0.3); ax.set_xlim(0, vmax)
ax.set_xlabel("step speed (\u00b5m/hour)")
ax.set_title(f"Pooled per-region step-speed (ridge; smoothing={SMOOTHING}, gap steps excluded={EXCLUDE_GAP_STEPS})")
plt.show()

# NEW — Cell 11 · Mean Squared Displacement (MSD)

### The idea

Pick a cell. Ask: *after a time lag τ, how far has it typically moved, squared?* Average that over all starting points and all cells in a region. Plotting **MSD vs τ** tells you the *style* of motion, which speed alone can't:

| MSD vs τ shape | log-log slope α | Motion type |
|----------------|-----------------|-------------|
| bends **down**, flattens to a plateau | α < 1 | **confined** — trapped in a region (your "parked" prediction) |
| straight line | α ≈ 1 | diffusive — a random walk |
| bends **up** | α > 1 (→2) | **directed** — persistent travel toward a target |

So MSD distinguishes a cell that *jiggles fast but goes nowhere* (confined, low α) from one that *creeps slowly toward the injury* (directed, high α) — exactly the inside-vs-outside question you wanted.

### How we compute it rigorously

- **Time-averaged, then ensemble-averaged.** For each track we compute its own MSD using *every* pair of points τ frames apart (time-averaging — far more data per cell than taking only one origin). Then we average those per-track curves within a region (the ensemble). This is the standard estimator for single-particle tracking.
- **Gap-aware.** This is the subtle bit, and the reason the code indexes points by **frame number**, not row order. If a cell is missing in frame 7, then the rows for frames 6 and 8 are *adjacent in the table* but are **2 frames apart in time**. Counting them as a 1-step would corrupt every lag. We build a `{frame → (x, y)}` lookup and only pair frames that genuinely are τ apart.
- **Real-time axis.** Lag is computed in frames, then converted to **minutes** per image, so the 30- and 40-min fish live on the same axis. We bin lag-minutes onto a common grid so the cadences pool.
- **Trust short lags only.** A track of length L has many pairs at lag 1 but very few at lag L−1, so long-lag MSD is noisy. We only fit α over short lags (`FIT_MAX_LAG_MIN`) and only plot lags where ≥3 tracks contribute.

> Read α as a *direction of evidence*, not a verdict — with this few tracks per region it's indicative. The shape of the curve (does it plateau? does it curl up?) is often more honest than the fitted number.

In [ ]:
# ---- knobs ---------------------------------------------------------------
MAX_LAG_FRAC    = 0.5     # use lags up to 50% of each track's span (longer = too few pairs)
MIN_SPOTS_MSD   = 5       # ignore very short tracks
LAG_BIN_MIN     = 20      # round lag-time to this grid so 30- & 40-min fish pool
FIT_MAX_LAG_MIN = 160     # fit alpha (log-log slope) over short lags only
MIN_TRACKS_PER_LAG = 3    # only plot a lag if at least this many tracks reach it
REGIONS_MSD = ["uninjured", "penumbra", "core"]
MOT_COLORS  = {"core": "darkred", "penumbra": "red", "uninjured": "blue", "outside": "grey"}

# per-image real-time interval (min) -- cadence differs across images
_p = spots.groupby("SOURCE_IMAGE").agg(_n=("FRAME", lambda s: int(s.max())+1),
                                       _t=("POSITION_T", "max"))
interval_min = (_p["_t"] / (_p["_n"] - 1) / 60).round().astype(int)

# one region label per track (majority of the fine label), tracked spots only
tracked = spots[spots["TRACK_ID"].notna()].copy()
region_of_track = tracked.groupby(KEYS)["region_fine"].agg(lambda s: s.value_counts().idxmax())

def time_averaged_msd(d, max_lag_frac=MAX_LAG_FRAC):
    # Gap-aware time-averaged MSD for ONE track.
    # Index by FRAME so missing frames are never treated as adjacent; for each
    # lag tau, average squared displacement over all real pairs exactly tau apart.
    d = d.dropna(subset=["POSITION_X", "POSITION_Y"])
    f = d["FRAME"].astype(int).to_numpy()
    pos = {int(fr): (x, y) for fr, x, y in zip(f, d["POSITION_X"], d["POSITION_Y"])}
    fmin, fmax = f.min(), f.max()
    span = fmax - fmin
    out = {}
    for lag in range(1, int(span * max_lag_frac) + 1):
        sq = [(pos[fr+lag][0]-pos[fr][0])**2 + (pos[fr+lag][1]-pos[fr][1])**2
              for fr in range(fmin, fmax - lag + 1) if fr in pos and fr+lag in pos]
        if sq:
            out[lag] = np.mean(sq)
    return out

# per-track curves -> long table
rows = []
for keys, d in tracked.groupby(KEYS):
    if len(d) < MIN_SPOTS_MSD:
        continue
    iv = interval_min[keys[0]]
    for lag, m in time_averaged_msd(d).items():
        rows.append((keys[0], keys[1], region_of_track.loc[keys], lag, lag*iv, m))
msd = pd.DataFrame(rows, columns=["SOURCE_IMAGE", "TRACK_ID", "region", "lag_frame", "lag_min", "msd_um2"])
msd["lag_bin"] = (np.round(msd["lag_min"] / LAG_BIN_MIN) * LAG_BIN_MIN).astype(int)

# ensemble-average per region & lag bin
ens = (msd.groupby(["region", "lag_bin"])["msd_um2"]
          .agg(["mean", "sem", "count"]).reset_index())

fig, ax = plt.subplots(figsize=(7.5, 6))
alpha_report = {}
for r in REGIONS_MSD:
    e = ens[(ens.region == r) & (ens["count"] >= MIN_TRACKS_PER_LAG)].sort_values("lag_bin")
    if e.empty:
        continue
    ax.errorbar(e["lag_bin"], e["mean"], yerr=e["sem"], marker="o",
                color=MOT_COLORS[r], capsize=3, lw=1.6, label=r)
    fit = e[(e["lag_bin"] <= FIT_MAX_LAG_MIN) & (e["mean"] > 0)]
    if len(fit) >= 3:                       # slope of log(MSD) vs log(tau) = alpha
        a, b = np.polyfit(np.log(fit["lag_bin"]), np.log(fit["mean"]), 1)
        alpha_report[r] = a
        xx = np.array([fit["lag_bin"].min(), fit["lag_bin"].max()], float)
        ax.plot(xx, np.exp(b) * xx**a, ls="--", lw=1, color=MOT_COLORS[r])

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("lag time \u03c4 (min)")
ax.set_ylabel("MSD  \u27e8\u0394r\u00b2\u27e9  (\u00b5m\u00b2)")
ax.set_title("Mean squared displacement per region (time-averaged, ensemble)")
ax.legend(title="region"); fig.tight_layout(); plt.show()

print("Fitted log-log slope \u03b1 (short lags):")
for r in REGIONS_MSD:
    if r in alpha_report:
        a = alpha_report[r]
        tag = "confined/sub-diffusive" if a < 0.85 else ("directed/super-diffusive" if a > 1.15 else "~diffusive")
        print(f"   {r:>9}: \u03b1 = {a:0.2f}   ({tag})")
print("\n   \u03b1<1 confined (plateau)   \u03b1\u22481 diffusive   \u03b1>1 directed (\u03b1\u21922 ballistic)")

# NEW — Cell 12 · Per-image trajectory maps

A direct, qualitative look that complements every aggregate above: **draw each cell's path**, one panel per fish. Each **dot is one frame**, coloured by the region that cell was in at that moment, so you can literally watch tracks change colour as they cross into the injured side. The faint grey line connects the dots in time; an **open circle marks the start**, a **star marks the end**.

Reading tips:
- The Y-axis is **inverted** so the picture matches the microscope image — since Step 04 puts the injury at the **top** (small Y), injury is at the top of each panel. A track drifting *upward* on screen is heading *toward* the injury.
- A confined cell looks like a **tight knot of dots**; a directed one looks like a **streak**. This is the same confined-vs-directed contrast as the MSD plot, but you can see it per cell.
- Watching where the colours change shows you boundary-crossers (your `is_migrator` cells from Cell 7) at a glance.

Techniques: `groupby("TRACK_ID")` to draw one path at a time, mapping a categorical column to colours via a dict + `.map`, `ax.set_aspect("equal")` so distances aren't distorted, and `ax.invert_yaxis()` for image convention.

In [ ]:
from matplotlib.lines import Line2D

# ---- knobs ---------------------------------------------------------------
NCOLS_TRAJ     = 3
MIN_SPOTS_TRAJ = 3        # skip 1-2 point fragments (no real path)
FINE_COLORS = {"core": "darkred", "penumbra": "red", "uninjured": "blue", "outside": "lightgrey"}

traj = spots[spots["TRACK_ID"].notna()].copy().sort_values(KEYS + ["FRAME"])
imgs = sorted(traj["SOURCE_IMAGE"].unique())
nrows = int(np.ceil(len(imgs) / NCOLS_TRAJ))
fig, axes = plt.subplots(nrows, NCOLS_TRAJ, figsize=(5 * NCOLS_TRAJ, 4.3 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, img in zip(axes, imgs):
    sub = traj[traj["SOURCE_IMAGE"] == img]
    for tid, d in sub.groupby("TRACK_ID"):
        if len(d) < MIN_SPOTS_TRAJ:
            continue
        x = d["POSITION_X"].to_numpy(); y = d["POSITION_Y"].to_numpy()
        ax.plot(x, y, "-", color="0.7", lw=0.8, zorder=1)                      # path in time order
        ax.scatter(x, y, c=d["region_fine"].map(FINE_COLORS).to_numpy(),       # one dot per frame
                   s=16, zorder=2, edgecolor="k", linewidth=0.2)
        ax.plot(x[0],  y[0],  marker="o", mfc="white", mec="k", ms=7,  zorder=3)  # start
        ax.plot(x[-1], y[-1], marker="*", mfc="k",     mec="k", ms=11, zorder=3)  # end
    ax.set_title(img.split("_")[2], fontsize=9)
    ax.set_aspect("equal"); ax.invert_yaxis()       # image convention: injury (small Y) at top
    ax.set_xlabel("X (\u00b5m)"); ax.set_ylabel("Y (\u00b5m)"); ax.tick_params(labelsize=7)

for ax in axes[len(imgs):]:
    ax.axis("off")

handles = [Line2D([], [], marker="o", ls="", color=FINE_COLORS[r], mec="k", label=r)
           for r in ["core", "penumbra", "uninjured", "outside"]]
handles += [Line2D([], [], marker="o", ls="", mfc="white", mec="k", label="start"),
            Line2D([], [], marker="*", ls="", mfc="k", mec="k", label="end")]
fig.legend(handles=handles, loc="lower center", ncol=6, frameon=False, fontsize=9)
fig.suptitle("Microglia trajectories per image  (injury at top; dot = one frame, coloured by region)",
             fontsize=13)
fig.tight_layout(rect=[0, 0.04, 1, 0.97]); plt.show()

# NEW — Binary view · injury core vs rest

The three-group plots above (Cells 10 & 11) keep the full `uninjured → penumbra → core` gradient. Sometimes you want the blunt **two-group** contrast instead: *is the cell in the injury or not?* These two cells add that, reusing the per-step (`steps`, from Cell 10) and per-track MSD (`msd`, from Cell 11) tables you already built — so **run Cells 10 and 11 first**.

**One choice, one switch — `PENUMBRA_AS_INJURY`.**

| Switch | "inside" group | "outside" group | Question it asks |
|--------|----------------|-----------------|------------------|
| `False` *(default — what you described)* | core (region 3) | uninjured + penumbra (1 + 2) | Do cells behave differently specifically in the **lesion core**? |
| `True` | penumbra + core (2 + 3) | uninjured (1) | Do cells behave differently anywhere on the **injured side**? |

Region 0 (outside annotated tissue / non-cord cells) is dropped either way. Set the switch the same in both cells so the ridge and the MSD describe the same split.

## Cell 13 — Binary step-speed ridge (injury core vs rest)

Same per-step speeds as Cell 10, but every step is relabelled to just two groups by the rule above, then drawn as two ridges. If cells park in the core, the "injury core" ridge should sit shifted toward low speed and its dashed median should fall to the left of the other.

Depends on `steps` from Cell 10. The `print` underneath gives the bare medians so you have a number, not just a shape.

In [ ]:
from scipy.stats import gaussian_kde

# ---- the one decision (keep identical in Cell 14) ------------------------
PENUMBRA_AS_INJURY = False   # False: inside = core only;  True: inside = penumbra + core
SMOOTHING = 0.3
OVERLAP   = 1.4

INSIDE  = ["core", "penumbra"] if PENUMBRA_AS_INJURY else ["core"]
OUTSIDE = [r for r in ["uninjured", "penumbra"] if r not in INSIDE]
IN_LABEL  = "injured side" if PENUMBRA_AS_INJURY else "injury core"
OUT_LABEL = "uninjured"    if PENUMBRA_AS_INJURY else "outside core"
BIN_COLORS = {IN_LABEL: "darkred", OUT_LABEL: "steelblue"}

def to_binary(fine):
    if fine in INSIDE:  return IN_LABEL
    if fine in OUTSIDE: return OUT_LABEL
    return None                       # region 0 (outside tissue) -> dropped

# `steps` was built in Cell 10 (per-step speeds, attributed to region at step start)
b = steps.copy()
b["bin_region"] = b["region_start"].map(to_binary)
b = b.dropna(subset=["bin_region"])

ORDER2 = [OUT_LABEL, IN_LABEL]        # top -> bottom of the ridge
vmax = np.nanpercentile(b["speed_um_hour"], 99)
xs   = np.linspace(0, vmax, 300)
n    = len(ORDER2)
dens, cnt = {}, {}
for r in ORDER2:
    v = b.loc[b["bin_region"] == r, "speed_um_hour"].dropna().values
    cnt[r] = len(v)
    dens[r] = gaussian_kde(v, bw_method=SMOOTHING)(xs) if (len(v) >= 2 and np.std(v) > 0) else np.zeros_like(xs)
gmax = max((d.max() for d in dens.values()), default=1.0) or 1.0

fig, ax = plt.subplots(figsize=(8, 3.8))
for idx, r in enumerate(ORDER2):
    base = n - 1 - idx
    d = dens[r] / gmax * OVERLAP
    ax.fill_between(xs, base, base + d, color=BIN_COLORS[r], alpha=0.8, zorder=idx + 2)
    ax.plot(xs, base + d, color="k", lw=0.8, zorder=idx + 2)
    v = b.loc[b["bin_region"] == r, "speed_um_hour"].dropna()
    if len(v):
        ax.plot([v.median()] * 2, [base, base + 0.18 * OVERLAP], color="k", lw=1.4, ls="--", zorder=idx + 2.1)
ax.set_yticks([n - 1 - i for i in range(n)])
ax.set_yticklabels([f"{r}\n(steps={cnt[r]})" for r in ORDER2])
for tick, r in zip(ax.get_yticklabels(), ORDER2):
    tick.set_color(BIN_COLORS[r]); tick.set_fontsize(9)
ax.set_ylim(-0.2, n - 1 + OVERLAP + 0.3); ax.set_xlim(0, vmax)
ax.set_xlabel("step speed (\u00b5m/hour)")
ax.set_title(f"Binary step-speed:  {IN_LABEL}  vs  {OUT_LABEL}")
plt.show()

print(b.groupby("bin_region")["speed_um_hour"].agg(["count", "median", "mean"]).round(1))

## Cell 14 — Binary MSD (injury core vs rest)

The same MSD curves as Cell 11, collapsed to the two groups. Read it the same way: a curve that **flattens / sits low with α < 1** = confined (the "parked" signature); a curve that **bends upward with α > 1** = directed. The cleanest evidence for your hypothesis is the "injury core" curve flattening *below* the other.

Depends on `msd` from Cell 11. Keep `PENUMBRA_AS_INJURY` the same as Cell 13.

In [ ]:
PENUMBRA_AS_INJURY = False   # keep identical to Cell 13
LAG_BIN_MIN        = 20
FIT_MAX_LAG_MIN    = 160
MIN_TRACKS_PER_LAG = 3

INSIDE  = ["core", "penumbra"] if PENUMBRA_AS_INJURY else ["core"]
OUTSIDE = [r for r in ["uninjured", "penumbra"] if r not in INSIDE]
IN_LABEL  = "injured side" if PENUMBRA_AS_INJURY else "injury core"
OUT_LABEL = "uninjured"    if PENUMBRA_AS_INJURY else "outside core"
BIN_COLORS = {IN_LABEL: "darkred", OUT_LABEL: "steelblue"}

def to_binary(fine):
    if fine in INSIDE:  return IN_LABEL
    if fine in OUTSIDE: return OUT_LABEL
    return None

# `msd` (per-track MSD curves) was built in Cell 11; relabel its 'region' to binary
mb = msd.copy()
mb["bin_region"] = mb["region"].map(to_binary)
mb = mb.dropna(subset=["bin_region"])
mb["lag_bin"] = (np.round(mb["lag_min"] / LAG_BIN_MIN) * LAG_BIN_MIN).astype(int)
ens2 = mb.groupby(["bin_region", "lag_bin"])["msd_um2"].agg(["mean", "sem", "count"]).reset_index()

fig, ax = plt.subplots(figsize=(7.5, 6))
for r in [IN_LABEL, OUT_LABEL]:
    e = ens2[(ens2.bin_region == r) & (ens2["count"] >= MIN_TRACKS_PER_LAG)].sort_values("lag_bin")
    if e.empty:
        continue
    ax.errorbar(e["lag_bin"], e["mean"], yerr=e["sem"], marker="o",
                color=BIN_COLORS[r], capsize=3, lw=1.8, label=r)
    fit = e[(e["lag_bin"] <= FIT_MAX_LAG_MIN) & (e["mean"] > 0)]
    if len(fit) >= 3:
        a, bb = np.polyfit(np.log(fit["lag_bin"]), np.log(fit["mean"]), 1)
        xx = np.array([fit["lag_bin"].min(), fit["lag_bin"].max()], float)
        ax.plot(xx, np.exp(bb) * xx**a, ls="--", lw=1, color=BIN_COLORS[r])
        tag = "confined" if a < 0.85 else ("directed" if a > 1.15 else "~diffusive")
        print(f"{r:>12}: \u03b1 = {a:0.2f}   ({tag})")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("lag time \u03c4 (min)")
ax.set_ylabel("MSD  \u27e8\u0394r\u00b2\u27e9  (\u00b5m\u00b2)")
ax.set_title(f"Binary MSD:  {IN_LABEL}  vs  {OUT_LABEL}")
ax.legend(); fig.tight_layout(); plt.show()

# Where this leaves you, and what to reach for next

**What the notebook now answers**
- **Q1 (recruitment):** Cells 4–6 — counts per region over time, per fish and pooled, with a paired within-fish view.
- **Q2 (parked vs roaming):** Cells 7–10 — per-track speed/confinement, a per-region metric, and step-speed distributions; Cell 11 (MSD) classifies the *style* of motion (confined vs directed); Cell 12 shows it cell-by-cell.

**Caveats carried through (don't lose these in a methods write-up)**
1. **Counts aren't area-normalised** — part of any injured/uninjured difference could be region size. Export region area (µm²) from Step 04 and divide.
2. **Pooling ignores fish-level grouping** — every "pooled" comparison treats tracks as independent. For a real statistic, use a mixed-effects model with fish as a random effect.
3. **Unlinked detections** (~11% here) count toward Q1 but not Q2 — `COUNT_ONLY_TRACKED` in Cell 4 lets you test sensitivity.
4. **Manual spots** are valid for counts/position, invalid for morphology — `is_manual` / `fraction_manual` keep them traceable.
5. **MSD with few tracks per region** is indicative, not conclusive — trust the curve shape over the fitted α, and revisit once you have the full 5–6 series.

**Natural next analyses** (each is a small extension of a table you already built):
- **Distance-to-injury over time** per track — needs the injury boundary geometry exported from Step 04 (the open README TODO: line vs polygon).
- **Convergence / flux across the boundary** — count net crossings using the `region_fine` changes already in `spots`.
- **Behavioural subtypes** — cluster tracks on `[speed, confinement, directedness, frac_core]` from `per_track`.

> One small documentation nit I noticed while checking: the repo **README prose** still says "injured = 1, uninjured = 2", but your current `04_draw_injury.ijm` actually writes **uninjured = 1, injured = 2, core = 3** — which is what this notebook (correctly) assumes, and what your sample data contains. Worth fixing the README line and the older `data_schema.md` so a future-you isn't misled.